In [2]:
import csv
import numpy as np

X_dados = []
y_dados = []

with open("PARTICIPANTES_2025.csv", "r", encoding="latin-1") as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=";")

    c= 0

    for linha in leitor:
        q3 = linha["Q003"].strip()
        q4 = linha["Q004"].strip()
        q7 = linha["Q007"].strip()
        c+= 1
        if q3 in "ABCDEF" and q4 in "ABCDEF" and q7 in "ABCDEFGHIJKLMNOPQ":
            X_dados.append([q3, q4])
            y_dados.append(q7)
        if c >= 100_000:
            break

print("Quantidade de participantes:", len(y_dados))
print("Primeiras entradas:", X_dados[:5])
print("Primeiras classes:", y_dados[:5])

Quantidade de participantes: 100000
Primeiras entradas: [['A', 'A'], ['C', 'B'], ['D', 'D'], ['D', 'D'], ['E', 'E']]
Primeiras classes: ['B', 'D', 'F', 'B', 'Q']


In [44]:
categorias = "ABCDEF"
categorias_renda = "ABCDEFGHIJKLMNOPQ"

# X com One-Hot Encoding
X = np.array([
    [
        np.eye(len(categorias))[categorias.index(q3)],
        np.eye(len(categorias))[categorias.index(q4)]
    ]
    for q3, q4 in X_dados
])

X = X.reshape(X.shape[0], -1)

# Y com One-Hot Encoding
Y = np.array([
    np.eye(len(categorias_renda))[categorias_renda.index(q7)]
    for q7 in y_dados
])

print("Formato de X:", X.shape)
print("Formato de Y:", Y.shape)

print("Primeira entrada de X:", X[0])
print("Primeira classe de Y:", Y[0])

print(X[:100])
print(Y[:100])

Formato de X: (100000, 12)
Formato de Y: (100000, 17)
Primeira entrada de X: [1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
Primeira classe de Y: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[[1. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]
 ...
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]]
[[0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [45]:
np.random.seed(42)

indices = np.random.permutation(len(Y))

limite = int(0.8 * len(Y))

indices_treino = indices[:limite]
indices_teste = indices[limite:]

X_treino = X[indices_treino]
Y_treino = Y[indices_treino]

X_teste = X[indices_teste]
Y_teste = Y[indices_teste]

print("Treinamento:", len(Y_treino))
print("Teste:", len(Y_teste))
print(X_treino)

Treinamento: 80000
Teste: 20000
[[0. 0. 0. ... 1. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 1. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]]


In [46]:
from machinelearning_mnpkllps.decision_tree import DecisionTreeClassification

# Converte Y_treino de One-Hot para classes de 1 a 17
Y_treino_classes = np.argmax(Y_treino, axis=1) + 1

# Cria e treina a árvore
arvore = DecisionTreeClassification(max_depth=5, min_samples=2)

arvore.adjust(X_treino, Y_treino_classes)

In [47]:
# Previsões da árvore (inteiros de 1 a 17)
previsoes = arvore.predict(X_teste)

# Converte as previsões para One-Hot Encoding
previsoes_one_hot = np.eye(len(categorias_renda))[
    previsoes.astype(int) - 1
]

# Comparação
print("Formato das previsões:", previsoes_one_hot.shape)
print("Formato dos valores reais:", Y_teste.shape)

print("\nPrevisões:")
print(previsoes_one_hot[:30])

print("\nValores reais:")
print(Y_teste[:30])

Formato das previsões: (20000, 17)
Formato dos valores reais: (20000, 17)

Previsões:
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 

In [49]:
for i in range(10):
    previsao = categorias_renda[np.argmax(previsoes_one_hot[i])]
    real = categorias_renda[np.argmax(Y_teste[i])]

    print(f"Exemplo {i+1}: Previsão = {previsao} | Valor real = {real}")

Exemplo 1: Previsão = B | Valor real = B
Exemplo 2: Previsão = G | Valor real = D
Exemplo 3: Previsão = C | Valor real = G
Exemplo 4: Previsão = B | Valor real = C
Exemplo 5: Previsão = B | Valor real = E
Exemplo 6: Previsão = B | Valor real = G
Exemplo 7: Previsão = C | Valor real = H
Exemplo 8: Previsão = B | Valor real = D
Exemplo 9: Previsão = C | Valor real = F
Exemplo 10: Previsão = G | Valor real = G


In [50]:
def metricas_multiclasse(Y_pred, Y_real):

    # Número de classes
    n_classes = Y_real.shape[1]

    # Acurácia: comparação das classes de cada observação
    acuracia = np.mean(
        np.argmax(Y_pred, axis=1) == np.argmax(Y_real, axis=1)
    )

    precisao_classes = []
    recall_classes = []
    F1_classes = []

    # Calcula as métricas individualmente para cada classe
    for c in range(n_classes):

        VP = np.sum((Y_pred[:, c] == 1) & (Y_real[:, c] == 1))
        FP = np.sum((Y_pred[:, c] == 1) & (Y_real[:, c] == 0))
        FN = np.sum((Y_pred[:, c] == 0) & (Y_real[:, c] == 1))

        precisao = VP / (VP + FP) if VP + FP > 0 else 0

        recall = VP / (VP + FN) if VP + FN > 0 else 0

        F1 = (
            2 * precisao * recall / (precisao + recall)
            if precisao + recall > 0 else 0
        )

        precisao_classes.append(precisao)
        recall_classes.append(recall)
        F1_classes.append(F1)

    # Média macro das métricas
    precisao_media = np.mean(precisao_classes)
    recall_medio = np.mean(recall_classes)
    F1_medio = np.mean(F1_classes)

    return {
        "Acurácia": acuracia,
        "Precisão (Macro)": precisao_media,
        "Recall (Macro)": recall_medio,
        "F1 (Macro)": F1_medio
    }

In [52]:
# Calcula as métricas
metricas = metricas_multiclasse(previsoes_one_hot, Y_teste)

# Imprime os resultados
print("\n===== MÉTRICAS DA ÁRVORE DE DECISÃO =====")

print(f"Acurácia:          {metricas['Acurácia']:.4f}")
print(f"Precisão (Macro):  {metricas['Precisão (Macro)']:.4f}")
print(f"Recall (Macro):    {metricas['Recall (Macro)']:.4f}")
print(f"F1 (Macro):        {metricas['F1 (Macro)']:.4f}")

print("=========================================")


===== MÉTRICAS DA ÁRVORE DE DECISÃO =====
Acurácia:          0.3111
Precisão (Macro):  0.0542
Recall (Macro):    0.1114
F1 (Macro):        0.0709
